# Revise one existing FDA indication

Select a historical replay, detect whether its indication changed, isolate target-specific evidence, and propose a reviewable patch without writing to `moalmanac-db`.

In [1]:
import json, os
from pathlib import Path
from pprint import pprint
from dotenv import find_dotenv, load_dotenv
from moalmanac_fda_curation.core.revise_indication import (
    assess_update, build_assessment_prompt, build_proposal_prompt,
    events_since, propose_revision, refresh_changelog,
)

## Load the Anthropic API key

In [2]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")
PROJECT_ROOT = Path(env_path).parent
print(f"Loaded ANTHROPIC_API_KEY from {env_path}")

Loaded ANTHROPIC_API_KEY from /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/.env


## Select a test case

`jemperli_multi_event` is a historical replay starting from exact 2021 wording: event 2 adds treatment restrictions and event 3 adds the explicit single-agent form. `opdivo_current` uses a real current MOAlmanac record and its document publication date.

In [3]:
TEST_CASE = "jemperli_multi_event"  # or "opdivo_current"
if TEST_CASE == "jemperli_multi_event":
    document = {"id": "doc:fda.jemperli", "drug_name_brand": "Jemperli", "drug_name_generic": "dostarlimab", "identification_number": 761174, "publication_date": "2021-04-22", "urls": ["https://www.accessdata.fda.gov/drugsatfda_docs/label/2021/761174s000lbl.pdf"]}
    indication = {
        "id": "ind:fda.jemperli:1", "document_id": "doc:fda.jemperli",
        "indication": "JEMPERLI is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with mismatch repair deficient (dMMR) recurrent or advanced endometrial cancer, as determined by an FDA-approved test, that has progressed on or following prior treatment with a platinum-containing regimen.",
        "initial_approval_date": "2021-04-22", "initial_approval_url": "https://www.accessdata.fda.gov/drugsatfda_docs/label/2021/761174s000lbl.pdf",
        "description": "The U.S. Food and Drug Administration granted approval to dostarlimab for the treatment of adult patients with mismatch repair deficient (dMMR) recurrent or advanced endometrial cancer, as determined by an FDA-approved test, that has progressed on or following prior treatment with a platinum-containing regimen.",
        "raw_biomarkers": "dMMR", "raw_cancer_type": "recurrent or advanced endometrial cancer", "raw_therapeutics": "Jemperli (dostarlimab)",
    }
    expected = "Events 2 and 3 contribute. The final form comes from event 3, so approval provenance should be 2023-07-31 and event 3's URL. Separate combination and solid-tumor indications must be excluded."
else:
    document = {"id": "doc:fda.opdivo", "drug_name_brand": "Opdivo", "drug_name_generic": "nivolumab", "identification_number": 125554, "publication_date": "2025-04-11", "urls": ["https://www.accessdata.fda.gov/drugsatfda_docs/label/2025/125554s129lbl.pdf"]}
    indication = {
        "id": "ind:fda.opdivo:1", "document_id": "doc:fda.opdivo",
        "indication": "OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.",
        "initial_approval_date": "2020-05-15", "initial_approval_url": "https://www.accessdata.fda.gov/drugsatfda_docs/label/2020/125554s080lbl.pdf",
        "description": "The U.S. Food and Drug Administration (FDA) granted approval to nivolumab in combination with ipilimumab for the first-line treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations.",
        "raw_biomarkers": "PD-L1 (>= 1%)", "raw_cancer_type": "non-small cell lung cancer", "raw_therapeutics": "nivolumab in combination with ipilimumab", "date_regular_approval": "2020-05-15", "date_accelerated_approval": None,
    }
    expected = "Select the March 2026 event changing FDA-approved to FDA-authorized; ignore unrelated events."
LAST_CURATION_DATE = document["publication_date"]
WORK_DIR = PROJECT_ROOT / "analyses/revisions" / TEST_CASE
DOCUMENT_JSON = WORK_DIR / "document.json"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DOCUMENT_JSON.write_text(json.dumps(document, indent=2) + "\n")
print(f"Expected: {expected}")
print(f"Cutoff from document publication_date: {LAST_CURATION_DATE}")
pprint(indication)

Expected: Events 2 and 3 contribute. The final form comes from event 3, so approval provenance should be 2023-07-31 and event 3's URL. Separate combination and solid-tumor indications must be excluded.
Cutoff from document publication_date: 2021-04-22
{'description': 'The U.S. Food and Drug Administration granted approval to '
                'dostarlimab for the treatment of adult patients with mismatch '
                'repair deficient (dMMR) recurrent or advanced endometrial '
                'cancer, as determined by an FDA-approved test, that has '
                'progressed on or following prior treatment with a '
                'platinum-containing regimen.',
 'document_id': 'doc:fda.jemperli',
 'id': 'ind:fda.jemperli:1',
 'indication': 'JEMPERLI is a programmed death receptor-1 (PD-1)-blocking '
               'antibody indicated for the treatment of adult patients with '
               'mismatch repair deficient (dMMR) recurrent or advanced '
               'endometrial c

## 1. Load and inspect post-curation events

Prefer the selected drug's existing changelog in `ai-assisted-gk-curation`. If the sibling artifact is unavailable, rebuild it locally.

In [4]:
changelog_filename = {
    "jemperli_multi_event": "Jemperli-bla761174-section1-changelog.json",
    "opdivo_current": "Opdivo-bla125554-section1-changelog.json",
}[TEST_CASE]
EXISTING_CHANGELOG_JSON = PROJECT_ROOT.parent / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/section1-changelogs" / changelog_filename
if EXISTING_CHANGELOG_JSON.exists():
    changelog = json.loads(EXISTING_CHANGELOG_JSON.read_text())
    print(f"Using existing changelog: {EXISTING_CHANGELOG_JSON}")
else:
    refreshed = refresh_changelog(DOCUMENT_JSON, WORK_DIR)
    changelog = refreshed["changelog"]
    print(f"Built changelog: {refreshed['json_path']}")
candidate_events = events_since(changelog, LAST_CURATION_DATE)
pprint(candidate_events)

downloading 20210422 http://www.accessdata.fda.gov/drugsatfda_docs/label/2021/761174s000lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/jemperli_multi_event/historical-labels/Jemperli-bla761174/2021-04-22-ORIG-1-761174s000lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/jemperli_multi_event/historical-labels/Jemperli-bla761174/2021-04-22-ORIG-1-761174s000lbl.md
downloading 20230209 http://www.accessdata.fda.gov/drugsatfda_docs/label/2023/761174s003s004lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/jemperli_multi_event/historical-labels/Jemperli-bla761174/2023-02-09-SUPPL-4-761174s003s004lbl.pdf
wrote /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/revisions/jemperli_multi_event/historical-labels/Jemperli-bla761174/2023-02-09-SUPPL-4-761174s003s004lbl.md
downloading 20230731 http://www.accessdata.fda.gov/drugsatfda_docs/label/202

## 2. Assess and isolate target-specific spans

The exact rendered prompt is printed first. Python then verifies the selected event numbers and target quotes against the canonical changelog.

In [5]:
print(build_assessment_prompt(indication, candidate_events))
assessment = assess_update(indication, candidate_events)
pprint(assessment["assessment"])
print("\nVerified scoped evidence:")
pprint(assessment["scoped_evidence"])
print("\nVerification errors:", assessment["verification_errors"])

# Task

Decide whether later versions of the label's Indications and Usage section
clinically changed this existing MOAlmanac FDA indication.

# Target indication

```json
{
  "id": "ind:fda.jemperli:1",
  "document_id": "doc:fda.jemperli",
  "indication": "JEMPERLI is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with mismatch repair deficient (dMMR) recurrent or advanced endometrial cancer, as determined by an FDA-approved test, that has progressed on or following prior treatment with a platinum-containing regimen.",
  "initial_approval_date": "2021-04-22",
  "initial_approval_url": "https://www.accessdata.fda.gov/drugsatfda_docs/label/2021/761174s000lbl.pdf",
  "description": "The U.S. Food and Drug Administration granted approval to dostarlimab for the treatment of adult patients with mismatch repair deficient (dMMR) recurrent or advanced endometrial cancer, as determined by an FDA-approved test, that has progressed on or follo

## 3. Propose from scoped evidence only

The proposal model sees only verified target-specific spans. Python deterministically sets approval date and URL from the latest verified target event.

In [6]:
if assessment["assessment"]["status"] == "updated" and assessment["verified"]:
    print(build_proposal_prompt(indication, assessment["scoped_evidence"]))
    proposal = propose_revision(indication, assessment)
    pprint(proposal)
else:
    print("No verified update to propose.")

# Task

Propose the minimal revision needed for this existing MOAlmanac indication to
reflect the verified target-specific evidence.

# Existing indication

```json
{
  "id": "ind:fda.jemperli:1",
  "document_id": "doc:fda.jemperli",
  "indication": "JEMPERLI is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with mismatch repair deficient (dMMR) recurrent or advanced endometrial cancer, as determined by an FDA-approved test, that has progressed on or following prior treatment with a platinum-containing regimen.",
  "initial_approval_date": "2021-04-22",
  "initial_approval_url": "https://www.accessdata.fda.gov/drugsatfda_docs/label/2021/761174s000lbl.pdf",
  "description": "The U.S. Food and Drug Administration granted approval to dostarlimab for the treatment of adult patients with mismatch repair deficient (dMMR) recurrent or advanced endometrial cancer, as determined by an FDA-approved test, that has progressed on or following pr

## Optional: save after curator inspection

In [ ]:
# output = WORK_DIR / "revision-proposal.json"
# output.write_text(json.dumps(proposal, indent=2) + "\n")
# output